# Model estimation and forecasting

## Define paths, returns, samples, and proxy for volatility

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA_PATH = Path("../data_clean/btc_daily_returns.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df = df.sort_values("Date")
df = df.set_index("Date")

df.head()

,price,log_return
Date,,
2016-01-02,433.437988,-0.206512
2016-01-03,430.010986,-0.793798
2016-01-04,433.091003,0.713712
2016-01-05,431.959991,-0.261490
2016-01-06,429.105011,-0.663130


In [5]:
print(df.info())

<class 'pandas.DataFrame'>
DatetimeIndex: 3803 entries, 2016-01-02 to 2026-05-31
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   price       3803 non-null   float64
 1   log_return  3803 non-null   float64
dtypes: float64(2)
memory usage: 89.1 KB
None


In [6]:
returns = df["log_return"].dropna()
returns.name = "BTC daily log return (%)"

returns.head()

Date
2016-01-02   -0.206512
2016-01-03   -0.793798
2016-01-04    0.713712
2016-01-05   -0.261490
2016-01-06   -0.663130
Name: BTC daily log return (%), dtype: float64

In [7]:
OUT_OF_SAMPLE_FRACTION = 0.20

n_obs = len(returns)
split_idx = int(np.floor((1 - OUT_OF_SAMPLE_FRACTION) * n_obs))

returns_in = returns.iloc[:split_idx].copy()
returns_out = returns.iloc[split_idx:].copy()

print(f"Total number of return observations: {n_obs}")
print(f"In-sample observations:             {len(returns_in)}")
print(f"Out-of-sample observations:         {len(returns_out)}")
print()
print(f"In-sample period:     {returns_in.index.min().date()} to {returns_in.index.max().date()}")
print(f"Out-of-sample period: {returns_out.index.min().date()} to {returns_out.index.max().date()}")

Total number of return observations: 3803
In-sample observations:             3042
Out-of-sample observations:         761

In-sample period:     2016-01-02 to 2024-04-30
Out-of-sample period: 2024-05-01 to 2026-05-31


In [10]:
# ------------------------------------------------------------
# Construct the out-of-sample realised volatility proxy
# ------------------------------------------------------------

mean_return_in = returns_in.mean()

volatility_proxy_out = (returns_out - mean_return_in) ** 2
volatility_proxy_out.name = "realised_volatility_proxy"

print(f"In-sample mean return: {mean_return_in:.6f}")
print()
print(volatility_proxy_out.head())

In-sample mean return: 0.162355

Date
2024-05-01    17.400140
2024-05-02     1.739983
2024-05-03    36.160564
2024-05-04     2.010098
2024-05-05     0.003136
Name: realised_volatility_proxy, dtype: float64


## Estimate ARCH(1)-normal benchmark model

In [ ]:
# imprt required packages
import arch
from arch import arch_model

## Diagnostic tests for ARCH(1)-Normal

In [29]:
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

In [30]:
DIAGNOSTIC_LAGS = [10, 20, 30]
ARCH_LM_LAGS = 10

std_resid = arch1_normal_fitted["standardized_residual"].dropna()
std_resid_sq = std_resid ** 2

std_resid.name = "standardized_residual"
std_resid_sq.name = "squared_standardized_residual"

print(f"Number of standardized residuals used for diagnostics: {len(std_resid)}")

Number of standardized residuals used for diagnostics: 3042


In [35]:
#Ljung-box test on standardized residuals
lb_std_resid = acorr_ljungbox(
    std_resid,
    lags=DIAGNOSTIC_LAGS,
    return_df=True
)

lb_std_resid = lb_std_resid.rename(columns={
    "lb_stat": "lb_stat_standardized_residual",
    "lb_pvalue": "lb_pvalue_standardized_residual"
})


#Ljung-Box test on squared standardized residuals
lb_std_resid_sq = acorr_ljungbox(
    std_resid_sq,
    lags=DIAGNOSTIC_LAGS,
    return_df=True
)

lb_std_resid_sq = lb_std_resid_sq.rename(columns={
    "lb_stat": "lb_stat_squared_standardized_residual",
    "lb_pvalue": "lb_pvalue_squared_standardized_residual"
})


#Combine diagnostic results
ljung_box_diagnostics_arch1_normal = pd.concat(
    [lb_std_resid, lb_std_resid_sq],
    axis=1
)

ljung_box_diagnostics_arch1_normal.index.name = "lag"

ljung_box_diagnostics_arch1_normal

,lb_stat_standardized_residual,lb_pvalue_standardized_residual,lb_stat_squared_standardized_residual,lb_pvalue_squared_standardized_residual
lag,,,,
10,13.016149,0.222770,58.423830,7.192892e-09
20,20.445164,0.430410,70.749570,1.373450e-07
30,32.874936,0.327995,79.463223,2.359155e-06


In [36]:
# ARCH-LM test for remaining ARCH effects

arch_lm_stat, arch_lm_pvalue, arch_lm_f_stat, arch_lm_f_pvalue = het_arch(
    std_resid,
    nlags=ARCH_LM_LAGS,
    ddof=arch1_normal_result.num_params
)

arch_lm_arch1_normal = pd.Series({
    "model": "ARCH(1)",
    "distribution": "Normal",
    "lags": ARCH_LM_LAGS,
    "lm_stat": arch_lm_stat,
    "lm_pvalue": arch_lm_pvalue,
    "f_stat": arch_lm_f_stat,
    "f_pvalue": arch_lm_f_pvalue
})

arch_lm_arch1_normal

model             ARCH(1)
distribution       Normal
lags                   10
lm_stat         53.991481
lm_pvalue             0.0
f_stat           5.482615
f_pvalue              0.0
dtype: object

## Generalized model estimation workflow of ARCH-family models

In [53]:
def fit_volatility_model(
    returns_series,
    model_name,
    volatility_model,
    p,
    o=0,
    q=0,
    distribution="normal",
    distribution_name=None,
    mean_model="Constant",
    rescale=False,
    cov_type="classic"
):
    """
    Estimate one ARCH-family volatility model.

    Parameters
    ----------
    returns_series : pandas.Series
        Return series used for model estimation.
    model_name : str
        Readable model name, for example "ARCH(1)" or "GARCH(1,1)".
    volatility_model : str
        Volatility model type used by the arch package:
        "ARCH", "GARCH", or "EGARCH".
    p : int
        Number of ARCH terms.
    o : int, optional
        Number of asymmetric terms. Relevant for EGARCH.
    q : int, optional
        Number of lagged conditional variance terms.
    distribution : str, optional
        Distribution code used by the arch package:
        "normal", "t", "ged", or "skewt".
    distribution_name : str, optional
        Readable distribution name used in output tables.
    mean_model : str, optional
        Conditional mean specification. We use "Constant".
    rescale : bool, optional
        Whether the arch package may internally rescale the data.
        We use False because returns are already expressed in percentages.
    cov_type : str, optional
        Covariance estimator for standard errors.

    Returns
    -------
    result : arch.univariate.base.ARCHModelResult
        Fitted model result.
    """

    model = arch_model(
        y=returns_series,
        mean=mean_model,
        vol=volatility_model,
        p=p,
        o=o,
        q=q,
        dist=distribution,
        rescale=rescale
    )

    result = model.fit(
        disp="off",
        cov_type=cov_type
    )

    # Store readable metadata for later tables.
    result.model_name = model_name
    result.distribution_name = distribution_name
    result.distribution_code = distribution
    result.volatility_model_name = volatility_model
    result.p = p
    result.o = o
    result.q = q

    return result

In [54]:
def extract_parameter_table(result):
    """
    Extract parameter estimates, standard errors, t-values and p-values
    from an estimated ARCH-family model.

    Parameters
    ----------
    result : arch.univariate.base.ARCHModelResult
        The fitted model result.

    Returns
    -------
    parameter_table : pandas.DataFrame
        Table with one row per estimated parameter.
    """

    parameter_table = pd.DataFrame({
        "model": result.model_name,
        "distribution": result.distribution_name,
        "parameter": result.params.index,
        "estimate": result.params.values,
        "std_error": result.std_err.values,
        "t_value": result.tvalues.values,
        "p_value": result.pvalues.values
    })

    return parameter_table

In [55]:
def extract_model_info(result):
    """
    Extract log-likelihood, AIC, BIC and related model information.

    Parameters
    ----------
    result : arch.univariate.base.ARCHModelResult
        The fitted model result.

    Returns
    -------
    model_info : pandas.Series
        Summary information for one estimated model.
    """

    model_info = pd.Series({
        "model": result.model_name,
        "distribution": result.distribution_name,
        "log_likelihood": result.loglikelihood,
        "aic": result.aic,
        "bic": result.bic,
        "n_obs": result.nobs,
        "num_params": result.num_params,
        "convergence_flag": result.convergence_flag
    })

    return model_info

In [56]:
def extract_fitted_series(result, returns_series):
    """
    Extract residuals, fitted conditional volatility, conditional variance,
    and standardized residuals from an estimated model.

    Parameters
    ----------
    result : arch.univariate.base.ARCHModelResult
        The fitted model result.
    returns_series : pandas.Series
        The return series used in estimation.

    Returns
    -------
    fitted : pandas.DataFrame
        DataFrame containing fitted quantities for one model.
    """

    residuals = result.resid
    conditional_volatility = result.conditional_volatility
    conditional_variance = conditional_volatility ** 2
    standardized_residuals = residuals / conditional_volatility

    fitted = pd.DataFrame({
        "return": returns_series,
        "residual": residuals,
        "conditional_volatility": conditional_volatility,
        "conditional_variance": conditional_variance,
        "standardized_residual": standardized_residuals
    })

    fitted = fitted.dropna()

    return fitted

In [57]:
def run_residual_diagnostics(
    result,
    fitted_series,
    diagnostic_lags=(10, 20, 30),
    arch_lm_lags=10
):
    """
    Run Ljung-Box tests and ARCH-LM test on standardized residuals.

    Parameters
    ----------
    result : arch.univariate.base.ARCHModelResult
        The fitted model result.
    fitted_series : pandas.DataFrame
        Output from extract_fitted_series().
    diagnostic_lags : tuple of int
        Lags used for Ljung-Box tests.
    arch_lm_lags : int
        Number of lags used in the ARCH-LM test.

    Returns
    -------
    diagnostics : dict
        Dictionary containing Ljung-Box and ARCH-LM diagnostic results.
    """

    std_resid = fitted_series["standardized_residual"].dropna()
    std_resid_sq = std_resid ** 2

    # Ljung-Box test on standardized residuals.
    lb_std_resid = acorr_ljungbox(
        std_resid,
        lags=list(diagnostic_lags),
        return_df=True
    )

    lb_std_resid = lb_std_resid.rename(columns={
        "lb_stat": "lb_stat_standardized_residual",
        "lb_pvalue": "lb_pvalue_standardized_residual"
    })

    # Ljung-Box test on squared standardized residuals.
    lb_std_resid_sq = acorr_ljungbox(
        std_resid_sq,
        lags=list(diagnostic_lags),
        return_df=True
    )

    lb_std_resid_sq = lb_std_resid_sq.rename(columns={
        "lb_stat": "lb_stat_squared_standardized_residual",
        "lb_pvalue": "lb_pvalue_squared_standardized_residual"
    })

    ljung_box_table = pd.concat(
        [lb_std_resid, lb_std_resid_sq],
        axis=1
    )

    ljung_box_table.index.name = "lag"
    ljung_box_table.insert(0, "model", result.model_name)
    ljung_box_table.insert(1, "distribution", result.distribution_name)

    # ARCH-LM test for remaining ARCH effects.
    arch_lm_stat, arch_lm_pvalue, arch_lm_f_stat, arch_lm_f_pvalue = het_arch(
        std_resid,
        nlags=arch_lm_lags,
        ddof=result.num_params
    )

    arch_lm_table = pd.Series({
        "model": result.model_name,
        "distribution": result.distribution_name,
        "lags": arch_lm_lags,
        "lm_stat": arch_lm_stat,
        "lm_pvalue": arch_lm_pvalue,
        "f_stat": arch_lm_f_stat,
        "f_pvalue": arch_lm_f_pvalue
    })

    diagnostics = {
        "ljung_box": ljung_box_table,
        "arch_lm": arch_lm_table
    }

    return diagnostics

In [58]:
test_result = fit_volatility_model(
    returns_series=returns_in,
    model_name="ARCH(1)",
    volatility_model="ARCH",
    p=1,
    o=0,
    q=0,
    distribution="normal"
)

test_parameter_table = extract_parameter_table(test_result)
test_model_info = extract_model_info(test_result)
test_fitted_series = extract_fitted_series(test_result, returns_in)
test_diagnostics = run_residual_diagnostics(
    result=test_result,
    fitted_series=test_fitted_series,
    diagnostic_lags=DIAGNOSTIC_LAGS,
    arch_lm_lags=ARCH_LM_LAGS
)

print(test_model_info)
print()
print(test_parameter_table)
print()
print(test_diagnostics["ljung_box"])
print()
print(test_diagnostics["arch_lm"])

model                    ARCH(1)
distribution                None
log_likelihood      -8255.223415
aic                  16516.44683
bic                 16534.507642
n_obs                       3042
num_params                     3
convergence_flag               0
dtype: object

     model distribution parameter   estimate  std_error    t_value  \
0  ARCH(1)         None        mu   0.185263   0.064805   2.858785   
1  ARCH(1)         None     omega  11.671028   0.363500  32.107330   
2  ARCH(1)         None  alpha[1]   0.164859   0.025733   6.406575   

         p_value  
0   4.252668e-03  
1  3.483574e-226  
2   1.488250e-10  

       model distribution  lb_stat_standardized_residual  \
lag                                                        
10   ARCH(1)         None                      13.016149   
20   ARCH(1)         None                      20.445164   
30   ARCH(1)         None                      32.874936   

     lb_pvalue_standardized_residual  lb_stat_squared_standard

## Define model and distribution grid

In [52]:
model_grid = [
    {
        "model_name": "ARCH(1)",
        "volatility_model": "ARCH",
        "p": 1,
        "o": 0,
        "q": 0
    },
    {
        "model_name": "ARCH(2)",
        "volatility_model": "ARCH",
        "p": 2,
        "o": 0,
        "q": 0
    },
    {
        "model_name": "ARCH(3)",
        "volatility_model": "ARCH",
        "p": 3,
        "o": 0,
        "q": 0
    },
    {
        "model_name": "GARCH(1,1)",
        "volatility_model": "GARCH",
        "p": 1,
        "o": 0,
        "q": 1
    },
    {
        "model_name": "EGARCH(1,1)",
        "volatility_model": "EGARCH",
        "p": 1,
        "o": 1,
        "q": 1
    }
]


distribution_grid = [
    {
        "distribution_name": "Normal",
        "distribution": "normal"
    },
    {
        "distribution_name": "Student-t",
        "distribution": "t"
    },
    {
        "distribution_name": "GED",
        "distribution": "ged"
    },
    {
        "distribution_name": "Skewed Student-t",
        "distribution": "skewt"
    }
]

In [63]:
estimation_grid = [
    {**model_spec, **distribution_spec}
    for model_spec in model_grid
    for distribution_spec in distribution_grid
]

estimation_grid_table = pd.DataFrame(estimation_grid)

estimation_grid_table

,model_name,volatility_model,p,o,q,distribution_name,distribution
0,ARCH(1),ARCH,1,0,0,Normal,normal
1,ARCH(1),ARCH,1,0,0,Student-t,t
2,ARCH(1),ARCH,1,0,0,GED,ged
3,ARCH(1),ARCH,1,0,0,Skewed Student-t,skewt
4,ARCH(2),ARCH,2,0,0,Normal,normal
5,ARCH(2),ARCH,2,0,0,Student-t,t
6,ARCH(2),ARCH,2,0,0,GED,ged
7,ARCH(2),ARCH,2,0,0,Skewed Student-t,skewt
8,ARCH(3),ARCH,3,0,0,Normal,normal
9,ARCH(3),ARCH,3,0,0,Student-t,t


In [68]:
model_results = {}
parameter_tables = []
model_info_rows = []
fitted_series_dict = {}
ljung_box_tables = []
arch_lm_rows = []
failed_models = []

for specification in estimation_grid:
    
    model_name = specification["model_name"]
    distribution_name = specification["distribution_name"]
    model_key = f"{model_name} | {distribution_name}"
    
    print(f"Estimating {model_key}...")
    
    
    result = fit_volatility_model(
        returns_series=returns_in,
        model_name=model_name,
        volatility_model=specification["volatility_model"],
        p=specification["p"],
        o=specification["o"],
        q=specification["q"],
        distribution=specification["distribution"],
        distribution_name=distribution_name
    )
        
    fitted_series = extract_fitted_series(
        result=result,
        returns_series=returns_in
    )
        
    diagnostics = run_residual_diagnostics(
        result=result,
        fitted_series=fitted_series,
        diagnostic_lags=DIAGNOSTIC_LAGS,
        arch_lm_lags=ARCH_LM_LAGS
    )
        
    model_results[model_key] = result
    fitted_series_dict[model_key] = fitted_series
        
    parameter_tables.append(extract_parameter_table(result))
    model_info_rows.append(extract_model_info(result))
    ljung_box_tables.append(diagnostics["ljung_box"])
    arch_lm_rows.append(diagnostics["arch_lm"])
        

print()
print("Model estimation loop completed.")
print(f"Successfully estimated models: {len(model_results)}")
print(f"Failed models:                 {len(failed_models)}")

Estimating ARCH(1) | Normal...
Estimating ARCH(1) | Student-t...
Estimating ARCH(1) | GED...
Estimating ARCH(1) | Skewed Student-t...
Estimating ARCH(2) | Normal...
Estimating ARCH(2) | Student-t...
Estimating ARCH(2) | GED...
Estimating ARCH(2) | Skewed Student-t...
Estimating ARCH(3) | Normal...
Estimating ARCH(3) | Student-t...
Estimating ARCH(3) | GED...
Estimating ARCH(3) | Skewed Student-t...
Estimating GARCH(1,1) | Normal...
Estimating GARCH(1,1) | Student-t...
Estimating GARCH(1,1) | GED...
Estimating GARCH(1,1) | Skewed Student-t...
Estimating EGARCH(1,1) | Normal...
Estimating EGARCH(1,1) | Student-t...
Estimating EGARCH(1,1) | GED...
Estimating EGARCH(1,1) | Skewed Student-t...

Model estimation loop completed.
Successfully estimated models: 20
Failed models:                 0


In [74]:
all_parameter_estimates = pd.concat(
    parameter_tables,
    ignore_index=True
)

model_comparison_table = pd.DataFrame(model_info_rows)

all_ljung_box_diagnostics = pd.concat(
    ljung_box_tables,
    axis=0
).reset_index()

all_arch_lm_diagnostics = pd.DataFrame(arch_lm_rows)

# Add a readable model key to the model comparison table.
model_comparison_table["model_key"] = (
    model_comparison_table["model"]
    + " | "
    + model_comparison_table["distribution"]
)

# Reorder columns for readability.
model_comparison_table = model_comparison_table[
    [
        "model_key",
        "model",
        "distribution",
        "log_likelihood",
        "aic",
        "bic",
        "n_obs",
        "num_params",
        "convergence_flag"
    ]
]

model_comparison_table

,model_key,model,distribution,log_likelihood,aic,bic,n_obs,num_params,convergence_flag
0,ARCH(1) | Normal,ARCH(1),Normal,-8255.223415,16516.446830,16534.507642,3042,3,0
1,ARCH(1) | Student-t,ARCH(1),Student-t,-7807.700251,15623.400502,15647.481584,3042,4,0
2,ARCH(1) | GED,ARCH(1),GED,-7795.423601,15598.847202,15622.928284,3042,4,0
3,ARCH(1) | Skewed Student-t,ARCH(1),Skewed Student-t,-7807.687218,15625.374437,15655.475789,3042,5,0
4,ARCH(2) | Normal,ARCH(2),Normal,-8231.129133,16470.258266,16494.339348,3042,4,0
5,ARCH(2) | Student-t,ARCH(2),Student-t,-7778.157261,15566.314521,15596.415874,3042,5,0
6,ARCH(2) | GED,ARCH(2),GED,-7767.120711,15544.241423,15574.342775,3042,5,0
7,ARCH(2) | Skewed Student-t,ARCH(2),Skewed Student-t,-7778.144682,15568.289364,15604.410987,3042,6,0
8,ARCH(3) | Normal,ARCH(3),Normal,-8208.079065,16426.158130,16456.259483,3042,5,0
9,ARCH(3) | Student-t,ARCH(3),Student-t,-7756.579269,15525.158537,15561.280160,3042,6,0


In [75]:
aic_ranking = model_comparison_table.sort_values(
    by="aic",
    ascending=True
).reset_index(drop=True)

bic_ranking = model_comparison_table.sort_values(
    by="bic",
    ascending=True
).reset_index(drop=True)

aic_ranking["aic_rank"] = np.arange(1, len(aic_ranking) + 1)
bic_ranking["bic_rank"] = np.arange(1, len(bic_ranking) + 1)

aic_ranking = aic_ranking[
    [
        "aic_rank",
        "model_key",
        "model",
        "distribution",
        "log_likelihood",
        "aic",
        "bic",
        "num_params",
        "n_obs"
    ]
]

bic_ranking = bic_ranking[
    [
        "bic_rank",
        "model_key",
        "model",
        "distribution",
        "log_likelihood",
        "aic",
        "bic",
        "num_params",
        "n_obs"
    ]
]

aic_ranking.head(10)

,aic_rank,model_key,model,distribution,log_likelihood,aic,bic,num_params,n_obs
0,1,"EGARCH(1,1) | Student-t","EGARCH(1,1)",Student-t,-7609.198140,15230.396279,15266.517902,6,3042
1,2,"EGARCH(1,1) | Skewed Student-t","EGARCH(1,1)",Skewed Student-t,-7609.128209,15232.256417,15274.398310,7,3042
2,3,"EGARCH(1,1) | GED","EGARCH(1,1)",GED,-7627.921539,15267.843079,15303.964701,6,3042
3,4,"GARCH(1,1) | Student-t","GARCH(1,1)",Student-t,-7638.871293,15287.742587,15317.843939,5,3042
4,5,"GARCH(1,1) | GED","GARCH(1,1)",GED,-7639.455974,15288.911947,15319.013299,5,3042
5,6,"GARCH(1,1) | Skewed Student-t","GARCH(1,1)",Skewed Student-t,-7638.861571,15289.723142,15325.844765,6,3042
6,7,ARCH(3) | GED,ARCH(3),GED,-7747.678180,15507.356360,15543.477983,6,3042
7,8,ARCH(3) | Student-t,ARCH(3),Student-t,-7756.579269,15525.158537,15561.280160,6,3042
8,9,ARCH(3) | Skewed Student-t,ARCH(3),Skewed Student-t,-7756.575703,15527.151407,15569.293300,7,3042
9,10,ARCH(2) | GED,ARCH(2),GED,-7767.120711,15544.241423,15574.342775,5,3042


In [76]:
bic_ranking.head(10)

,bic_rank,model_key,model,distribution,log_likelihood,aic,bic,num_params,n_obs
0,1,"EGARCH(1,1) | Student-t","EGARCH(1,1)",Student-t,-7609.198140,15230.396279,15266.517902,6,3042
1,2,"EGARCH(1,1) | Skewed Student-t","EGARCH(1,1)",Skewed Student-t,-7609.128209,15232.256417,15274.398310,7,3042
2,3,"EGARCH(1,1) | GED","EGARCH(1,1)",GED,-7627.921539,15267.843079,15303.964701,6,3042
3,4,"GARCH(1,1) | Student-t","GARCH(1,1)",Student-t,-7638.871293,15287.742587,15317.843939,5,3042
4,5,"GARCH(1,1) | GED","GARCH(1,1)",GED,-7639.455974,15288.911947,15319.013299,5,3042
5,6,"GARCH(1,1) | Skewed Student-t","GARCH(1,1)",Skewed Student-t,-7638.861571,15289.723142,15325.844765,6,3042
6,7,ARCH(3) | GED,ARCH(3),GED,-7747.678180,15507.356360,15543.477983,6,3042
7,8,ARCH(3) | Student-t,ARCH(3),Student-t,-7756.579269,15525.158537,15561.280160,6,3042
8,9,ARCH(3) | Skewed Student-t,ARCH(3),Skewed Student-t,-7756.575703,15527.151407,15569.293300,7,3042
9,10,ARCH(2) | GED,ARCH(2),GED,-7767.120711,15544.241423,15574.342775,5,3042


In [78]:
best_aic_model_key = aic_ranking.loc[0, "model_key"]
best_bic_model_key = bic_ranking.loc[0, "model_key"]

best_aic_result = model_results[best_aic_model_key]
best_bic_result = model_results[best_bic_model_key]

print(f"Best model according to AIC: {best_aic_model_key}")
print(f"Best model according to BIC: {best_bic_model_key}")

Best model according to AIC: EGARCH(1,1) | Student-t
Best model according to BIC: EGARCH(1,1) | Student-t


In [79]:
best_aic_parameters = all_parameter_estimates[
    (all_parameter_estimates["model"] == aic_ranking.loc[0, "model"])
    & (all_parameter_estimates["distribution"] == aic_ranking.loc[0, "distribution"])
].reset_index(drop=True)

best_bic_parameters = all_parameter_estimates[
    (all_parameter_estimates["model"] == bic_ranking.loc[0, "model"])
    & (all_parameter_estimates["distribution"] == bic_ranking.loc[0, "distribution"])
].reset_index(drop=True)

best_aic_parameters

,model,distribution,parameter,estimate,std_error,t_value,p_value
0,"EGARCH(1,1)",Student-t,mu,0.134580,0.034318,3.921519,8.799240e-05
1,"EGARCH(1,1)",Student-t,omega,0.084190,0.024639,3.416973,6.332167e-04
2,"EGARCH(1,1)",Student-t,alpha[1],0.257994,0.035204,7.328598,2.325728e-13
3,"EGARCH(1,1)",Student-t,gamma[1],0.039412,0.015544,2.535476,1.122946e-02
4,"EGARCH(1,1)",Student-t,beta[1],0.994430,0.003858,257.736107,0.000000e+00
5,"EGARCH(1,1)",Student-t,nu,2.522879,0.149520,16.873159,7.090153e-64


In [82]:
all_ljung_box_diagnostics = all_ljung_box_diagnostics.copy()
all_arch_lm_diagnostics = all_arch_lm_diagnostics.copy()

all_ljung_box_diagnostics["model_key"] = (
    all_ljung_box_diagnostics["model"]
    + " | "
    + all_ljung_box_diagnostics["distribution"]
)

all_arch_lm_diagnostics["model_key"] = (
    all_arch_lm_diagnostics["model"]
    + " | "
    + all_arch_lm_diagnostics["distribution"]
)

# Reorder columns for readability.
all_ljung_box_diagnostics = all_ljung_box_diagnostics[
    [
        "model_key",
        "model",
        "distribution",
        "lag",
        "lb_stat_standardized_residual",
        "lb_pvalue_standardized_residual",
        "lb_stat_squared_standardized_residual",
        "lb_pvalue_squared_standardized_residual"
    ]
]

all_arch_lm_diagnostics = all_arch_lm_diagnostics[
    [
        "model_key",
        "model",
        "distribution",
        "lags",
        "lm_stat",
        "lm_pvalue",
        "f_stat",
        "f_pvalue"
    ]
]

all_ljung_box_diagnostics.head()

,model_key,model,distribution,lag,lb_stat_standardized_residual,lb_pvalue_standardized_residual,lb_stat_squared_standardized_residual,lb_pvalue_squared_standardized_residual
0,ARCH(1) | Normal,ARCH(1),Normal,10,13.016149,0.222770,58.423830,7.192892e-09
1,ARCH(1) | Normal,ARCH(1),Normal,20,20.445164,0.430410,70.749570,1.373450e-07
2,ARCH(1) | Normal,ARCH(1),Normal,30,32.874936,0.327995,79.463223,2.359155e-06
3,ARCH(1) | Student-t,ARCH(1),Student-t,10,13.589735,0.192541,48.642844,4.735308e-07
4,ARCH(1) | Student-t,ARCH(1),Student-t,20,21.710607,0.356418,54.346416,5.138712e-05


In [85]:
SUMMARY_DIAGNOSTIC_LAG = 10
SIGNIFICANCE_LEVEL = 0.05

ljung_box_lag_summary = all_ljung_box_diagnostics[
    all_ljung_box_diagnostics["lag"] == SUMMARY_DIAGNOSTIC_LAG
].copy()

diagnostic_summary_table = ljung_box_lag_summary.merge(
    all_arch_lm_diagnostics,
    on=["model_key", "model", "distribution"],
    how="left"
)

diagnostic_summary_table = diagnostic_summary_table[
    [
        "model_key",
        "model",
        "distribution",
        "lag",
        "lb_pvalue_standardized_residual",
        "lb_pvalue_squared_standardized_residual",
        "lags",
        "lm_pvalue",
        "f_pvalue"
    ]
]

diagnostic_summary_table = diagnostic_summary_table.rename(columns={
    "lag": "ljung_box_lag",
    "lags": "arch_lm_lags"
})

diagnostic_summary_table.head()

,model_key,model,distribution,ljung_box_lag,lb_pvalue_standardized_residual,lb_pvalue_squared_standardized_residual,arch_lm_lags,lm_pvalue,f_pvalue
0,ARCH(1) | Normal,ARCH(1),Normal,10,0.222770,7.192892e-09,10,4.870021e-08,4.075741e-08
1,ARCH(1) | Student-t,ARCH(1),Student-t,10,0.192541,4.735308e-07,10,7.320645e-07,6.371546e-07
2,ARCH(1) | GED,ARCH(1),GED,10,0.201666,1.357125e-07,10,2.893917e-07,2.480198e-07
3,ARCH(1) | Skewed Student-t,ARCH(1),Skewed Student-t,10,0.193073,4.665667e-07,10,7.252278e-07,6.268855e-07
4,ARCH(2) | Normal,ARCH(2),Normal,10,0.265757,4.720792e-05,10,5.699690e-05,5.282448e-05


In [89]:
diagnostic_summary_table["no_residual_autocorrelation_5pct"] = (
    diagnostic_summary_table["lb_pvalue_standardized_residual"] >= SIGNIFICANCE_LEVEL
)

diagnostic_summary_table["no_squared_residual_autocorrelation_5pct"] = (
    diagnostic_summary_table["lb_pvalue_squared_standardized_residual"] >= SIGNIFICANCE_LEVEL
)

diagnostic_summary_table["no_remaining_arch_effects_5pct"] = (
    diagnostic_summary_table["lm_pvalue"] >= SIGNIFICANCE_LEVEL
)

diagnostic_summary_table["passes_all_diagnostics_5pct"] = (
    diagnostic_summary_table["no_residual_autocorrelation_5pct"]
    & diagnostic_summary_table["no_squared_residual_autocorrelation_5pct"]
    & diagnostic_summary_table["no_remaining_arch_effects_5pct"]
)

aic_rank_lookup = aic_ranking[["model_key", "aic_rank"]].copy()
bic_rank_lookup = bic_ranking[["model_key", "bic_rank"]].copy()

model_diagnostic_comparison_table = (
    model_comparison_table
    .merge(aic_rank_lookup, on="model_key", how="left")
    .merge(bic_rank_lookup, on="model_key", how="left")
    .merge(
        diagnostic_summary_table[
            [
                "model_key",
                "lb_pvalue_standardized_residual",
                "lb_pvalue_squared_standardized_residual",
                "lm_pvalue",
                "f_pvalue",
                "passes_all_diagnostics_5pct"
            ]
        ],
        on="model_key",
        how="left"
    )
)

model_diagnostic_comparison_table = model_diagnostic_comparison_table[
    [
        "model_key",
        "model",
        "distribution",
        "aic_rank",
        "bic_rank",
        "log_likelihood",
        "aic",
        "bic",
        "lb_pvalue_standardized_residual",
        "lb_pvalue_squared_standardized_residual",
        "lm_pvalue",
        "f_pvalue",
        "passes_all_diagnostics_5pct",
        "num_params",
        "n_obs"
    ]
]

model_diagnostic_comparison_table = model_diagnostic_comparison_table.sort_values(
    by="aic_rank"
).reset_index(drop=True)

model_diagnostic_comparison_table.head(10)

,model_key,model,distribution,aic_rank,bic_rank,log_likelihood,aic,bic,lb_pvalue_standardized_residual,lb_pvalue_squared_standardized_residual,lm_pvalue,f_pvalue,passes_all_diagnostics_5pct,num_params,n_obs
0,"EGARCH(1,1) | Student-t","EGARCH(1,1)",Student-t,1,1,-7609.198140,15230.396279,15266.517902,0.005613,0.679074,0.688218,0.687582,False,6,3042
1,"EGARCH(1,1) | Skewed Student-t","EGARCH(1,1)",Skewed Student-t,2,2,-7609.128209,15232.256417,15274.398310,0.005585,0.681558,0.690798,0.689933,False,7,3042
2,"EGARCH(1,1) | GED","EGARCH(1,1)",GED,3,3,-7627.921539,15267.843079,15303.964701,0.015288,0.943531,0.945542,0.945480,False,6,3042
3,"GARCH(1,1) | Student-t","GARCH(1,1)",Student-t,4,4,-7638.871293,15287.742587,15317.843939,0.013608,0.911959,0.912583,0.912553,False,5,3042
4,"GARCH(1,1) | GED","GARCH(1,1)",GED,5,5,-7639.455974,15288.911947,15319.013299,0.014120,0.898040,0.897839,0.897792,False,5,3042
5,"GARCH(1,1) | Skewed Student-t","GARCH(1,1)",Skewed Student-t,6,6,-7638.861571,15289.723142,15325.844765,0.013568,0.912076,0.912782,0.912660,False,6,3042
6,ARCH(3) | GED,ARCH(3),GED,7,7,-7747.678180,15507.356360,15543.477983,0.223484,0.233547,0.215768,0.214452,True,6,3042
7,ARCH(3) | Student-t,ARCH(3),Student-t,8,8,-7756.579269,15525.158537,15561.280160,0.222176,0.366184,0.346682,0.345395,True,6,3042
8,ARCH(3) | Skewed Student-t,ARCH(3),Skewed Student-t,9,9,-7756.575703,15527.151407,15569.293300,0.222037,0.366092,0.346851,0.345283,True,7,3042
9,ARCH(2) | GED,ARCH(2),GED,10,10,-7767.120711,15544.241423,15574.342775,0.204927,0.003397,0.003187,0.003075,False,5,3042


In [90]:
# ------------------------------------------------------------
# Block 9.7: Identify diagnostically adequate models
# ------------------------------------------------------------

diagnostically_adequate_models = model_diagnostic_comparison_table[
    model_diagnostic_comparison_table["passes_all_diagnostics_5pct"]
].copy()

diagnostically_adequate_models = diagnostically_adequate_models.sort_values(
    by="aic"
).reset_index(drop=True)

print(f"Number of models passing all diagnostics at 5%: {len(diagnostically_adequate_models)}")

diagnostically_adequate_models

Number of models passing all diagnostics at 5%: 3


,model_key,model,distribution,aic_rank,bic_rank,log_likelihood,aic,bic,lb_pvalue_standardized_residual,lb_pvalue_squared_standardized_residual,lm_pvalue,f_pvalue,passes_all_diagnostics_5pct,num_params,n_obs
0,ARCH(3) | GED,ARCH(3),GED,7,7,-7747.678180,15507.356360,15543.477983,0.223484,0.233547,0.215768,0.214452,True,6,3042
1,ARCH(3) | Student-t,ARCH(3),Student-t,8,8,-7756.579269,15525.158537,15561.280160,0.222176,0.366184,0.346682,0.345395,True,6,3042
2,ARCH(3) | Skewed Student-t,ARCH(3),Skewed Student-t,9,9,-7756.575703,15527.151407,15569.293300,0.222037,0.366092,0.346851,0.345283,True,7,3042


In [91]:
best_in_sample_model_keys = sorted(
    set([best_aic_model_key, best_bic_model_key])
)

best_in_sample_diagnostics = model_diagnostic_comparison_table[
    model_diagnostic_comparison_table["model_key"].isin(best_in_sample_model_keys)
].copy()

best_in_sample_diagnostics

,model_key,model,distribution,aic_rank,bic_rank,log_likelihood,aic,bic,lb_pvalue_standardized_residual,lb_pvalue_squared_standardized_residual,lm_pvalue,f_pvalue,passes_all_diagnostics_5pct,num_params,n_obs
0,"EGARCH(1,1) | Student-t","EGARCH(1,1)",Student-t,1,1,-7609.19814,15230.396279,15266.517902,0.005613,0.679074,0.688218,0.687582,False,6,3042


In [92]:
diagnostic_display_table = model_diagnostic_comparison_table.copy()

columns_to_round = [
    "log_likelihood",
    "aic",
    "bic",
    "lb_pvalue_standardized_residual",
    "lb_pvalue_squared_standardized_residual",
    "lm_pvalue",
    "f_pvalue"
]

diagnostic_display_table[columns_to_round] = diagnostic_display_table[columns_to_round].round(4)

diagnostic_display_table.head(10)

,model_key,model,distribution,aic_rank,bic_rank,log_likelihood,aic,bic,lb_pvalue_standardized_residual,lb_pvalue_squared_standardized_residual,lm_pvalue,f_pvalue,passes_all_diagnostics_5pct,num_params,n_obs
0,"EGARCH(1,1) | Student-t","EGARCH(1,1)",Student-t,1,1,-7609.1981,15230.3963,15266.5179,0.0056,0.6791,0.6882,0.6876,False,6,3042
1,"EGARCH(1,1) | Skewed Student-t","EGARCH(1,1)",Skewed Student-t,2,2,-7609.1282,15232.2564,15274.3983,0.0056,0.6816,0.6908,0.6899,False,7,3042
2,"EGARCH(1,1) | GED","EGARCH(1,1)",GED,3,3,-7627.9215,15267.8431,15303.9647,0.0153,0.9435,0.9455,0.9455,False,6,3042
3,"GARCH(1,1) | Student-t","GARCH(1,1)",Student-t,4,4,-7638.8713,15287.7426,15317.8439,0.0136,0.9120,0.9126,0.9126,False,5,3042
4,"GARCH(1,1) | GED","GARCH(1,1)",GED,5,5,-7639.4560,15288.9119,15319.0133,0.0141,0.8980,0.8978,0.8978,False,5,3042
5,"GARCH(1,1) | Skewed Student-t","GARCH(1,1)",Skewed Student-t,6,6,-7638.8616,15289.7231,15325.8448,0.0136,0.9121,0.9128,0.9127,False,6,3042
6,ARCH(3) | GED,ARCH(3),GED,7,7,-7747.6782,15507.3564,15543.4780,0.2235,0.2335,0.2158,0.2145,True,6,3042
7,ARCH(3) | Student-t,ARCH(3),Student-t,8,8,-7756.5793,15525.1585,15561.2802,0.2222,0.3662,0.3467,0.3454,True,6,3042
8,ARCH(3) | Skewed Student-t,ARCH(3),Skewed Student-t,9,9,-7756.5757,15527.1514,15569.2933,0.2220,0.3661,0.3469,0.3453,True,7,3042
9,ARCH(2) | GED,ARCH(2),GED,10,10,-7767.1207,15544.2414,15574.3428,0.2049,0.0034,0.0032,0.0031,False,5,3042
